In [2]:
import torch
import patcher
import os
from matplotlib import pyplot as plt

context = patcher.Context('../db', 1)

os.makedirs('../images/database_test', exist_ok=True)

def draw(request, filename, method='load', rng=None):
    x = patcher.load(context, [request]) if method == 'load' else patcher.load_climate(context, [request])
    x = x[0]
    x = x[tuple([-1] * (x.dim() - 2) + [slice(None), slice(None)])]
    plt.figure(figsize=(6, 4))
    if method == 'load':
        if rng == 'auto':
            plt.pcolormesh(x)
        elif 'lsm' in filename:
            plt.pcolormesh(x, vmin=0, vmax=1)
        else:
            plt.pcolormesh(x, vmin=-15, vmax=15)
    else:
        plt.pcolormesh(x)
    plt.colorbar()
    plt.savefig(f'../images/database_test/{filename}.png')
    plt.close()

In [3]:
x, y = ds[0]
print(x.keys())
print(y.keys())

dict_keys(['t2m_local', 't2m_regional', 't2m_global', 'lsm_local', 'lsm_regional', 'lsm_global', 'inm_t2m_regional', 'inm_t2m_global', 'lat_local', 'lat_regional', 'lat_global', 'lon_local', 'lon_regional', 'lon_global', 'year_local', 'year_regional', 'year_global', 'day_local', 'day_regional', 'day_global', 'inm_lead_time_regional', 'inm_lead_time_global'])
dict_keys(['t2m'])


In [2]:
draw(patcher.Request('t2m', 0, 0, '19800101', 1440, 361, 1, 1, 1), 'era')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440 // 4, 361 // 4 + 1, 1, 4, 1), 'era_1regional')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440 // 16, 361 // 16 + 1, 1, 16, 1), 'era_2global')
draw(patcher.Request('t2m', 720, 0, '19800101', 1440, 361, 1, 1, 1), 'era_shifted_lon')
draw(patcher.Request('t2m', 0, 90, '19800101', 1440, 361, 1, 1, 1), 'era_shifted_lat')
draw(patcher.Request('t2m', 0, -90, '19800101', 1440, 361, 1, 1, 1), 'era_shifted_lat_negative')
draw(patcher.Request('t2m', -70, 200, '19800101', 507, 99, 1, 1, 1), 'era_cropped')
draw(patcher.Request('t2m', 0, 0, '20260430', 1440, 361, 1, 1, 1), 'era_last')
draw(patcher.Request('t2m', 0, 0, '20260401', 1440, 361, 30, 1, 1), 'era_last_time_size')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440, 361, 1, 1, 30), 'era_aggregated')
draw(patcher.Request('t2m', 0, 0, '19910101', 1440, 361, 1, 1, 365*30 + 8), 'era_aggregated_large') # Должен быть 0 или очень близко
draw(patcher.Request('t2m', 0, 0, '19800101', 1440, 361, 1, 1, 1), 'era_climate', 'load_climate')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440 // 4, 361 // 4 + 1, 1, 4, 1), 'era_climate_1regional', 'load_climate')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440 // 16, 361 // 16 + 1, 1, 16, 1), 'era_climate_2global', 'load_climate')
draw(patcher.Request('t2m', 0, 0, '19800101', 1440, 361, 1, 1, 30), 'era_aggregate_climate', 'load_climate')

draw(patcher.Request('inm_t2m', 0, 0, '19920101', 360, 91, 1, 1, 1, 1), 'forecast')
draw(patcher.Request('inm_t2m', 0, 0, '20250101', 360, 91, 1, 1, 1, 1), 'forecast_operative')
draw(patcher.Request('inm_t2m', 0, 0, '19920101', 360, 91, 1, 1, 1, 2), 'forecast_null') # Должен быть пустым
draw(patcher.Request('inm_t2m', 0, 0, '19920101', 360 // 4, 91 // 4 + 1, 1, 4, 1, 1), 'forecast_1global')
draw(patcher.Request('inm_t2m', 0, 0, '19920101', 360, 91, 1, 1, 1, 1), 'forecast_climate', 'load_climate')
draw(patcher.Request('inm_t2m', 0, 0, '19920101', 360, 91, 1, 1, 30, 1), 'forecast_aggregated')

draw(patcher.Request('lsm', 0, 0, '19800101', 1440, 361, 1, 1, 1), 'lsm')
draw(patcher.Request('lsm', 0, 0, '19800101', 1440, 361, 1, 1, 30), 'lsm_aggregated')
draw(patcher.Request('lsm', 0, 0, '19800101', 1440, 361, 1, 1, 1), 'lsm_climate', 'load_climate') # Должны быть нули

In [2]:
import sys
sys.path.append('../db')
from content import ELEMENTS

for element in ELEMENTS:
    if 'inm_' in element:
        draw(patcher.Request(element, 0, 0, '19920101', 360, 91, 1, 1, 1, 1), f'element_{element}', rng='auto')
    else:
        draw(patcher.Request(element, 0, 0, '19800101', 1440, 361, 1, 1, 1), f'element_{element}', rng='auto')

In [1]:
from database import PatchDataset
from matplotlib import pyplot as plt

ds = PatchDataset(
    input_variables=['t2m', 'lsm', 'inm_t2m', 'lat', 'lon', 'year', 'day', 'inm_lead_time', 'inm_day', 'inm_lat', 'inm_lon'],
    target_variables=['t2m', 'day', 'lat', 'lon'],
    time_range=('19910101', '20201231'),
    mask='snow',
    epoch_size=1000,
    batch_size=8,
    era_scales=[
        {'id': 'local', 'xSize': 16, 'ySize': 16, 'tSize': 7, 'xyStep': 1, 'tStep': 1},
        {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 6, 'xyStep': 4, 'tStep': 7},
        {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 16, 'tStep': 30},
    ],
    inm_scales=[
        {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 6, 'xyStep': 1, 'tStep': 1},
        {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 7},
    ],
    target_scale={'xSize': 16, 'ySize': 16, 'tSize': 6, 'xyStep': 4, 'tStep': 1},
)

x, y = ds[0]

print('X')
for key in x.keys():
    print(key, x[key].shape)

print('\nY')
for key in y.keys():
    print(key, y[key].shape)

X
t2m_local torch.Size([8, 7, 16, 16])
t2m_regional torch.Size([8, 6, 16, 16])
t2m_global torch.Size([8, 4, 16, 16])
lsm_local torch.Size([8, 7, 16, 16])
lsm_regional torch.Size([8, 6, 16, 16])
lsm_global torch.Size([8, 4, 16, 16])
inm_t2m_regional torch.Size([8, 10, 6, 16, 16])
inm_t2m_global torch.Size([8, 10, 4, 16, 16])
lat_local torch.Size([8, 16])
lat_regional torch.Size([8, 16])
lat_global torch.Size([8, 16])
lon_local torch.Size([8, 16])
lon_regional torch.Size([8, 16])
lon_global torch.Size([8, 16])
year_local torch.Size([8, 7])
year_regional torch.Size([8, 6])
year_global torch.Size([8, 4])
day_local torch.Size([8, 7])
day_regional torch.Size([8, 6])
day_global torch.Size([8, 4])
inm_lead_time_regional torch.Size([8, 6])
inm_lead_time_global torch.Size([8, 4])
inm_day_regional torch.Size([8, 6])
inm_day_global torch.Size([8, 4])
inm_lat_regional torch.Size([8, 16])
inm_lat_global torch.Size([8, 16])
inm_lon_regional torch.Size([8, 16])
inm_lon_global torch.Size([8, 16])

Y
t2

In [2]:
for level in ['local', 'regional', 'global']:
    for key in ['lon', 'lat', 'year', 'day']:
        print(f'{key}_{level}', x[f'{key}_{level}'][0, ...])
print('inm_lead_time_regional', x['inm_lead_time_regional'][0, ...])
print('inm_lead_time_global', x['inm_lead_time_global'][0, ...])

print()
for match_key in ['day', 'lat', 'lon']:
    print(f'inm_{match_key}', x[f'inm_{match_key}_regional'][0, ...])
    print(f'target_{match_key}', y[match_key][0, ...]) # Должен совпадать с предыдущей строчкой

lon_local tensor([248.2500, 248.5000, 248.7500, 249.0000, 249.2500, 249.5000, 249.7500,
        250.0000, 250.2500, 250.5000, 250.7500, 251.0000, 251.2500, 251.5000,
        251.7500, 252.0000])
lat_local tensor([57.7500, 58.0000, 58.2500, 58.5000, 58.7500, 59.0000, 59.2500, 59.5000,
        59.7500, 60.0000, 60.2500, 60.5000, 60.7500, 61.0000, 61.2500, 61.5000])
year_local tensor([2004., 2004., 2004., 2004., 2004., 2004., 2004.])
day_local tensor([63., 64., 65., 66., 67., 68., 69.])
lon_regional tensor([242., 243., 244., 245., 246., 247., 248., 249., 250., 251., 252., 253.,
        254., 255., 256., 257.])
lat_regional tensor([51., 52., 53., 54., 55., 56., 57., 58., 59., 60., 61., 62., 63., 64.,
        65., 66.])
year_regional tensor([2004., 2004., 2004., 2004., 2004., 2004.])
day_regional tensor([28., 35., 42., 49., 56., 63.])
lon_global tensor([216., 220., 224., 228., 232., 236., 240., 244., 248., 252., 256., 260.,
        264., 268., 272., 276.])
lat_global tensor([24., 28., 32., 

In [5]:
def draw_red_square(data, N):
    center_y, center_x = data.shape[0] / 2, data.shape[1] / 2
    y_start = center_y - N / 2
    x_start = center_x - N / 2

    rect = plt.Rectangle((x_start, y_start), N, N, linewidth=1, edgecolor='red', facecolor='none')
    plt.gca().add_patch(rect)

# несовпадают из-за разных осреднений по времени и это нормально
plt.pcolormesh(x['t2m_local'][0, 0, :, :])
plt.savefig(f'../images/database_test/database_t2m_local.png')
plt.close()

plt.pcolormesh(x['t2m_regional'][0, 0, :, :])
plt.savefig(f'../images/database_test/database_t2m_regional.png')
plt.close()

plt.pcolormesh(x['t2m_global'][0, 0, :, :])
plt.savefig(f'../images/database_test/database_t2m_global.png')
plt.close()

plt.pcolormesh(x['lsm_local'][0, 0, :, :])
plt.savefig(f'../images/database_test/database_lsm_local.png')
plt.close()

plt.pcolormesh(x['lsm_regional'][0, 0, :, :])
draw_red_square(x['lsm_regional'][0, 0, :, :], 4)
plt.savefig(f'../images/database_test/database_lsm_regional.png')
plt.close()

plt.pcolormesh(x['lsm_global'][0, 0, :, :])
draw_red_square(x['lsm_global'][0, 0, :, :], 4)
draw_red_square(x['lsm_global'][0, 0, :, :], 1)
plt.savefig(f'../images/database_test/database_lsm_global.png')
plt.close()

plt.pcolormesh(x['inm_t2m_regional'][0, 0, 0, :, :])
plt.savefig(f'../images/database_test/database_inm_t2m_regional.png')
plt.close()

plt.pcolormesh(x['inm_t2m_global'][0, 0, 0, :, :])
plt.savefig(f'../images/database_test/database_inm_t2m_global.png')
plt.close()

In [4]:
# speed test
from tqdm.notebook import tqdm

for x in tqdm(ds.loader):
    pass

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 